# The ESEM sandbox, run end to end

This model asks who would build a power station, rather than what a system ought to
contain. Each firm faces an income it cannot predict, values that income below its
average because it is uncertain, and commits only when what it expects to earn covers
what the plant costs to own. Nobody in it is obliged to build anything.

A least-cost model answers the other question, and the distance between the two
answers is what this notebook is about.

Everything here is illustrative. The fleet is stylised, the weather is synthetic, and
the system is one region with no transmission in it.

The runs below are shorter than the model's own defaults so that a cell finishes while
you are looking at it: 18 possible futures rather than 45, and eight years rather than
20. Totals here are not comparable with the 20-year figures in the repository.


In [ ]:
!pip install -q git+https://github.com/MarkusMannheim/esem_sandbox.git  # skip if you have it

from esem_sandbox.config import load_settings
from esem_sandbox.core.forward import cell_plan
from esem_sandbox.core.simulate import run, MERCHANT, ESEM
from esem_sandbox import plots
import numpy as np

settings = load_settings()
# A mild weather year and the lull-on-heat year, so investors see both a quiet
# future and a stressed one. Keeping only mild years would have them planning for
# weather the model then never gives them.
FAST = tuple(c for c in cell_plan(settings) if c.shape_year in (0, 4))
TICKS, SEED = 8, 20260904
print(f'{len(cell_plan(settings))} futures in the full lattice, {len(FAST)} here')


## 1. One region, one year

Plant is stacked cheapest first and the price is set by the last unit needed. In most
hours that is tens of dollars. In a handful it is thousands, and the cell below counts
how few hours those are and how much of the year's price they carry.


In [ ]:
from esem_sandbox.core.dispatch import dispatch_year
from esem_sandbox.core.weather import generate_bundle

bundle = generate_bundle(settings.weather['seed'], settings.weather['shape_years'])
shape = bundle['demand_shape'][4]          # the year with a wind lull over a heatwave
year = dispatch_year(settings, 2026, shape * (12500 / shape.max()),
                     bundle['wind_cf'][4], bundle['solar_cf'][4])

hours_above = int((year.price >= 300).sum())
print(f'{hours_above} hours of 8,760 priced at or above $300/MWh')
print(f'they carry {year.price[year.price >= 300].sum() / year.price.sum():.0%} '
      'of the year total')
print(f'unserved energy {year.unserved_mwh.sum() / 1000:.3f} GWh')


In [ ]:
plots.price_duration({'the lull-on-heat year': year}, 'duration.png')
from IPython.display import Image
Image('duration.png')


## 2. What a plant has to earn before anybody builds it

This is the model's subject. Each firm prices a project against 18 possible futures,
and values that spread of outcomes below its average because it is uncertain. The gap
between what a peaker costs to own and what a firm demands before building one is
mostly caution.


In [ ]:
from esem_sandbox.core.agents import PRODUCER
from esem_sandbox.core.clearing import cara_certainty_equivalent, cara_coefficient
from esem_sandbox.core.esem import long_run_cost_per_mw_year
from esem_sandbox.core.forward import EntryState, forward_view
from esem_sandbox.core.investment import residual_exposure

base = run(settings, ticks=1, seed=SEED, cells=FAST, leg=MERCHANT)
view = forward_view(settings, base.fleet, bundle, year=2026, peak_mw=12500.0,
                    entry=EntryState(), cells=FAST)
tech = settings.tech('ocgt')
fixed = long_run_cost_per_mw_year(tech, tech.wacc, 0.0)
print(f'a peaker costs {fixed:>12,.0f} $/MW-year to own\n')
rents = view.lifetime_rent(tech)
for agent in [a for a in base.roster if a.kind == PRODUCER]:
    exposure = residual_exposure(settings, tech.life_years)
    a = cara_coefficient(agent.risk_aversion, exposure, settings)
    ce = cara_certainty_equivalent(rents, view.weights, a)
    print(f'{agent.name:>20} demands {fixed + max(0.0, rents @ view.weights - ce):>12,.0f} '
          f'$/MW-year')


The difference between those two figures is what the firm gives up in exchange for
certainty. Nobody receives it, and it is most of the bar.

That is why a long contract can cause plant to be built. Selling output forward
removes the uncertainty rather than paying for it, so the bar falls without anybody
handing the plant money.


In [ ]:
for cover in (0.0, 0.3, 0.6, 0.9):
    exposure = residual_exposure(settings, tech.life_years,
                                 award_years=tech.life_years, award_cover=cover)
    a = cara_coefficient(0.6, exposure, settings)
    ce = cara_certainty_equivalent(rents, view.weights, a)
    bar = fixed + max(0.0, rents @ view.weights - ce)
    print(f'{cover:>4.0%} of output sold forward   bar {bar:>12,.0f} $/MW-year')


## 3. What the market builds, and why the model brackets it

The amount built depends on something the model cannot settle: whether investors can
see each other. Two rules bound it. Under the first, nobody observes anybody, so every
firm prices its project against a market that does not contain the others' projects
and all of them build. Under the second, everybody observes everybody instantly, so
the first decision of a year removes the scarcity rent the rest were counting on.

Real investors are neither. The gap between the two rules is wider than most of the
policy effects this model measures, which is why capacity adequacy is argued about.


In [ ]:
for rule in ('simultaneous', 'sequential'):
    r = run(settings, ticks=TICKS, seed=SEED, cells=FAST, leg=MERCHANT,
            investment=rule)
    built = sum(b.capacity_mw for t in r.ticks for b in t.builds)
    print(f'{rule:>14}: built {built:>7,.0f} MW   '
          f'firm {r.ticks[-1].firm_capacity_mw:>7,.0f} MW   '
          f'unserved {r.total_unserved_gwh:>7.2f} GWh')


Read the two rows against each other rather than either on its own. Where they
disagree, the model is telling you it does not know.

When somebody says a capacity scheme is worth having, or is not, ask first how much
they think the market would have built without it. That is where the argument lives.


## 4. Reliability is not proportional to capacity

Take firm plant away in small steps and dispatch the same weather year each time. The
load in the worst hours is steep, so each megawatt removed exposes many more hours
than the last one did.


In [ ]:
from dataclasses import replace

VARIABLE = {'wind', 'solar', 'rooftop'}
print(f"{'firm capacity':>16}{'unserved, GWh':>16}{'times the base':>16}")
base_gwh = None
for scale in (1.00, 0.97, 0.94, 0.91, 0.85):
    fleet = tuple(u if u.technology in VARIABLE
                  else replace(u, capacity_mw=u.capacity_mw * scale,
                               must_run_mw=u.must_run_mw * scale)
                  for u in settings.fleet)
    res = dispatch_year(replace(settings, fleet=fleet), 2026,
                        shape * (12500 / shape.max()), bundle['wind_cf'][4],
                        bundle['solar_cf'][4])
    firm = sum(u.capacity_mw * u.availability for u in fleet
               if u.technology not in VARIABLE)
    gwh = float(res.unserved_mwh.sum()) / 1000
    base_gwh = base_gwh or gwh
    print(f'{firm:>13,.0f} MW{gwh:>16.2f}{gwh / base_gwh:>15.1f}x')


That is why the disagreement in section 3 matters. A few per cent of firm capacity is
the difference between a system that sheds nothing and one that sheds a lot.


## 5. The scheme, on one weather sequence

Both legs draw one weather sequence from one seed, so the difference between them is
the mechanism. A leg that drew its own weather would report the difference between two
climates as the effect of a policy.


In [ ]:
legs = {leg: run(settings, ticks=TICKS, seed=SEED, cells=FAST, leg=leg)
        for leg in (MERCHANT, ESEM)}
assert legs[MERCHANT].draw == legs[ESEM].draw   # one weather sequence

standard = settings.reliability['standard_use_fraction']
print(f"{'year':>6}{'merchant':>16}{'with the scheme':>18}{'lane MW':>10}")
for a, b in zip(legs[MERCHANT].ticks, legs[ESEM].ticks):
    print(f'{a.year:>6}{a.unserved_fraction/standard:>13.2f}x'
          f'{b.unserved_fraction/standard:>17.2f}x{b.lane_volume_mw:>10.0f}')


Look at the first two or three years. The legs are identical however much the lane
bought, because the plant it paid for has not been built yet. A procurement scheme is
an instrument about the future and it cannot fix a year that arrives before its plant
does.


In [ ]:
m, e = legs[MERCHANT], legs[ESEM]
bill = m.consumer_cost(settings) - e.consumer_cost(settings)
real = m.resource_cost(settings) - e.resource_cost(settings)
print(f'the bill moves          {bill/1e9:>8,.2f} bn   (positive means consumers pay less)')
print(f'the resource cost moves {real/1e9:>8,.2f} bn   (positive means the economy gives up less)')
print(f'the difference is a transfer of {(bill-real)/1e9:,.2f} bn')


Most of a bill is a payment from consumers to producers. Capacity pushes the pool
price down and moves money between them without saving any, so a comparison that
showed only the bill would report that movement as a benefit. Read both lines.


In [ ]:
plots.dashboard(legs, settings, 'dashboard.png')
Image('dashboard.png')


## Changing one thing

Each of these is one line, and each changes something the sections above measured.

- Contract length: `run(..., leg=ESEM)` against settings loaded with
  `{'esem': {'contract_tenor_years': 6}}`.
- Risk aversion: `{'investment': {'risk_premium': 0.0}}`. It moves the fleet in most
  weather draws, and not always in the same direction, because it raises each
  investor's own bar and also raises the bar it assumes of everybody else.
- The seed: `run(..., seed=20260101)`. What survives a different weather draw, and
  what was a property of this one?
- One year by hand: print the offer stack, the sorted prices, the hours at each rung
  of the demand-response ladder and the megawatt hours at the price cap. Everything
  in this model is that arithmetic repeated.

If a word here was unfamiliar,
[GLOSSARY.md](https://github.com/MarkusMannheim/esem_sandbox/blob/main/GLOSSARY.md)
defines every term the model uses, in the order you meet it.
